# 03. Approach 2 — ResNet-18 with a Frozen Pretrained Backbone

**Project:** Road Sign Recognition Using Deep Learning (Cambodian Traffic Signs, CamTSD)
**Classes:** 26 canonical traffic-sign categories

This notebook trains **Approach 2: transfer learning with a frozen backbone**, and evaluates it once on the held-out test set.

- **Backbone:** torchvision ResNet-18 with ImageNet weights (`IMAGENET1K_V1`), about 11.2M parameters, **all frozen**. It is used as a fixed feature extractor: every image becomes a 512-dimensional feature vector.
- **Classifier:** the original 1000-class layer is replaced by a new `Linear(512, 26)`. This is the **only trainable layer** (512 × 26 + 26 = 13,338 parameters).
- **Frozen BatchNorm:** the backbone's BatchNorm layers stay in evaluation mode even during training, so their ImageNet statistics are not overwritten by our batches. `FrozenBackboneResNet` in `src/models/resnet.py` handles this.

**Fair comparison with Approach 1 (SmallCNN from scratch):** same fixed split and manifests, transforms, seed, batch size, optimizer settings, scheduler, early stopping and maximum epochs. Only the model changes.

All logic comes from the project modules in `src/`. The notebook only wires them together:

| Concern | Module |
|---|---|
| Classes, seed, image size | `src/config.py`, `src/seed.py` |
| Dataset, transforms, DataLoaders | `src/data/dataset.py`, `src/data/transforms.py` |
| Model | `src/models/resnet.py` → `create_resnet18_frozen` |
| Training loop, best-model selection | `src/training/train.py` → `train_model` |
| Checkpoints | `src/training/checkpoint.py` |

**Protocol (no leakage):**
- Train on the **train** split only (2,153 crops).
- Select the model and adjust the learning rate using the **validation** split only (454 crops).
- Use the **test** split (463 crops) exactly once, after training, on the best-validation checkpoint.
- No hyperparameter tuning is done in this notebook. The fixed settings are listed in the configuration cell.

**Runtime:** Google Colab with a GPU (`Runtime → Change runtime type → GPU`).

## 1. Setup

### 1.1 Environment, Google Drive and repository

**Expected Google Drive layout** (edit `DRIVE_PROJECT_DIR` if yours differs):

```
MyDrive/Road_Sign_Recognition_CNN/
└── dataset/
    └── processed/                 # crops produced by notebook 01
        ├── train/<CLASS_NAME>/*.jpg
        ├── val/<CLASS_NAME>/*.jpg
        └── test/<CLASS_NAME>/*.jpg
```
A `dataset/processed.zip` whose top-level folder is `processed/` also works.

The code and the portable split manifests (`dataset/splits/*.csv`, with crop paths relative to `dataset/processed/`) come from the GitHub repository and are used as-is. Results and checkpoints are written to Drive, outside Git.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# ---- Edit these if your setup differs ------------------------------------
REPO_URL = "https://github.com/RothRothRoth/DeepLearning_CNN.git"
REPO_BRANCH = "main"
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Road_Sign_Recognition_CNN")
# --------------------------------------------------------------------------

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    PROJECT_ROOT = Path("/content/DeepLearning_CNN")
    if (PROJECT_ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True
        )

    SOURCE_DATASET_DIR = DRIVE_PROJECT_DIR / "dataset"         # processed crops on Drive (not in Git)
    PROCESSED_DIR = Path("/content/camtsd_dataset/processed")  # fast local copy used for training
    OUTPUT_ROOT = DRIVE_PROJECT_DIR / "results" / "runs" / "resnet18_frozen"
else:
    # Local execution (e.g. checking the notebook on a workstation).
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    SOURCE_DATASET_DIR = PROJECT_ROOT / "dataset"
    PROCESSED_DIR = SOURCE_DATASET_DIR / "processed"
    OUTPUT_ROOT = PROJECT_ROOT / "results" / "runs" / "resnet18_frozen"  # gitignored

# Optional override for where run outputs go.
OUTPUT_ROOT = Path(os.environ.get("RESNET18_FROZEN_OUTPUT_ROOT", OUTPUT_ROOT))

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip() or "unknown"

print(f"Running in Colab:   {IN_COLAB}")
print(f"Project root:       {PROJECT_ROOT}")
print(f"Git commit:         {GIT_COMMIT}")
print(f"Source dataset dir: {SOURCE_DATASET_DIR}")
print(f"Output root:        {OUTPUT_ROOT}")

### 1.2 Requirements

In [ ]:
import importlib

requirements = PROJECT_ROOT / "requirements.txt"
if requirements.exists() and requirements.read_text().strip():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
else:
    print("requirements.txt is empty; using the packages preinstalled in Colab.")

for pkg in ["torch", "torchvision", "numpy", "pandas", "PIL", "sklearn", "matplotlib"]:
    module = importlib.import_module(pkg)
    print(f"  {pkg:12s} {getattr(module, '__version__', '')}")

### 1.3 Project imports, PyTorch and GPU

In [ ]:
import dataclasses
import json
import platform
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from torch import nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

from src.config import BATCH_SIZE, CLASS_NAMES, DEFAULT_SEED, IMAGE_SIZE, NUM_CLASSES, get_paths
from src.seed import set_seed
from src.data.dataset import get_dataloaders, resolve_crop_path

try:
    from src.models.resnet import (
        RESNET18_WEIGHTS,
        count_parameters,
        create_resnet18_frozen,
        get_trainable_parameters,
    )
    from src.training import get_device, load_checkpoint, train_model, validate
except ImportError as exc:
    raise ImportError(
        "ResNet-18 / training modules not found. The cloned repository is older than "
        "commit 41ba0e4: push the local commits to GitHub, then re-run the setup cell."
    ) from exc

assert NUM_CLASSES == 26 and len(CLASS_NAMES) == 26, "Expected exactly 26 classes"

device = get_device()
print(f"PyTorch:      {torch.__version__}")
print(f"torchvision:  {torchvision.__version__}")
print(f"CUDA:         {torch.cuda.is_available()} (CUDA {torch.version.cuda}, cuDNN {torch.backends.cudnn.version()})")
print(f"Device:       {device}")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(device)
    print(f"GPU:          {props.name}")
    print(f"GPU memory:   {props.total_memory / 1024**3:.1f} GB")
elif IN_COLAB:
    print("WARNING: no GPU detected. Use Runtime → Change runtime type → GPU.")

HARDWARE = {
    "device": str(device),
    "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
    "gpu_memory_gb": round(torch.cuda.get_device_properties(device).total_memory / 1024**3, 2)
    if device.type == "cuda"
    else None,
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "colab": IN_COLAB,
}

### 1.4 Experiment configuration

These settings are fixed and **identical to Approach 1**, so the comparison changes only the model. They are not the result of tuning. Every value is saved with the checkpoint and the results.

In [ ]:
SEED = DEFAULT_SEED  # 42

HPARAMS = {
    "model_name": "resnet18_frozen",
    "approach": "2_resnet18_frozen_backbone",
    "pretrained": True,
    "pretrained_weights": f"ResNet18_Weights.{RESNET18_WEIGHTS.name}",
    "trainable_layers": "fc only (Linear 512 -> 26); backbone + BatchNorm statistics frozen",
    "num_classes": NUM_CLASSES,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": 2,
    "optimizer": "Adam (trainable parameters only)",
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "loss": "CrossEntropyLoss",
    "max_epochs": 40,
    "scheduler": "ReduceLROnPlateau",
    "scheduler_mode": "max",
    "scheduler_factor": 0.5,
    "scheduler_patience": 3,
    "monitor": "val_acc",
    "early_stopping_patience": 10,
    "seed": SEED,
}

RUN_NAME = time.strftime("%Y%m%d-%H%M%S") + f"_seed{SEED}"
RUN_DIR = OUTPUT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
FIGURE_DIR = RUN_DIR / "figures"
METRICS_DIR = RUN_DIR / "metrics"
for d in (CHECKPOINT_DIR, FIGURE_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / "resnet18_frozen_best.pt"
with open(RUN_DIR / "config.json", "w") as f:
    json.dump({"run_name": RUN_NAME, "git_commit": GIT_COMMIT, "hyperparameters": HPARAMS}, f, indent=2)

print(f"Run directory: {RUN_DIR}")
print(json.dumps(HPARAMS, indent=2))

## 2. Dataset

### 2.1 Stage the processed crops

Reading thousands of small JPEGs directly from Drive is slow, so in Colab the `processed/` crops are copied once to the local runtime disk. Locally, the repository's `dataset/` folder is used directly.

In [ ]:
if IN_COLAB:
    drive_processed = SOURCE_DATASET_DIR / "processed"
    drive_zip = SOURCE_DATASET_DIR / "processed.zip"
    if PROCESSED_DIR.is_dir():
        print(f"Crops already staged at {PROCESSED_DIR}")
    elif drive_processed.is_dir():
        print(f"Copying {drive_processed} -> {PROCESSED_DIR} ...")
        t0 = time.perf_counter()
        shutil.copytree(drive_processed, PROCESSED_DIR)
        print(f"Done in {time.perf_counter() - t0:.0f}s")
    elif drive_zip.is_file():
        print(f"Extracting {drive_zip} -> {PROCESSED_DIR.parent} ...")
        shutil.unpack_archive(str(drive_zip), str(PROCESSED_DIR.parent))
    else:
        raise FileNotFoundError(
            f"No processed crops found. Expected {drive_processed}/ or {drive_zip} on Google Drive."
        )

if not PROCESSED_DIR.is_dir():
    raise FileNotFoundError(f"Processed crops not found at {PROCESSED_DIR}")
print("Splits present:", sorted(p.name for p in PROCESSED_DIR.iterdir() if p.is_dir()))

### 2.2 Load and verify the leak-free split manifests

The committed manifests in `dataset/splits/` are used directly. Each `crop_path` is a portable POSIX path relative to `dataset/processed/` (e.g. `train/KM_POST/<crop>.jpg`), resolved with `resolve_crop_path`, the same function `TrafficSignDataset` uses.

The only path override is `processed_dir`, which points at the local copy of the crops. This cell checks the split sizes, class coverage and zero parent-image overlap, and confirms that every crop file exists.

In [ ]:
EXPECTED_SIZES = {"train": 2153, "val": 454, "test": 463}

paths = dataclasses.replace(
    get_paths(str(PROJECT_ROOT)),     # manifests: <repo>/dataset/splits
    processed_dir=PROCESSED_DIR,      # crops: local copy (Colab) or <repo>/dataset/processed
    models_dir=CHECKPOINT_DIR,
    results_dir=RUN_DIR,
)
print(f"Manifests: {paths.splits_dir}")
print(f"Crops:     {paths.processed_dir}")

split_dfs = {s: pd.read_csv(paths.splits_dir / f"{s}.csv") for s in ("train", "val", "test")}

# Split sizes
for split, df in split_dfs.items():
    assert len(df) == EXPECTED_SIZES[split], f"{split}: {len(df)} rows, expected {EXPECTED_SIZES[split]}"
    assert (df["split"] == split).all(), f"{split}: rows with a different split label"

# Classes
for split, df in split_dfs.items():
    unknown = set(df["class_name"]) - set(CLASS_NAMES)
    assert not unknown, f"{split}: unknown classes {unknown}"
assert set(split_dfs["train"]["class_name"]) == set(CLASS_NAMES), "train split must cover all 26 classes"

# Zero parent-image overlap (no leakage)
parents = {s: set(df["parent_filename"]) for s, df in split_dfs.items()}
assert not parents["train"] & parents["val"], "train/val parent overlap"
assert not parents["train"] & parents["test"], "train/test parent overlap"
assert not parents["val"] & parents["test"], "val/test parent overlap"

# Every crop exists (resolve_crop_path also rejects non-portable paths)
missing = [
    p for df in split_dfs.values() for p in df["crop_path"]
    if not resolve_crop_path(paths.processed_dir, p).is_file()
]
assert not missing, f"{len(missing)} crop files missing, e.g. {missing[:3]}"

print("\nManifest checks passed.")
for split, df in split_dfs.items():
    print(f"  {split:5s}: {len(df):5d} crops from {len(parents[split]):5d} parent images")

class_dist = pd.DataFrame(
    {s: df["class_name"].value_counts() for s, df in split_dfs.items()}
).reindex(CLASS_NAMES).fillna(0).astype(int)
print("\nClass distribution:")
print(class_dist.to_string())

### 2.3 DataLoaders

Built with the project's `get_dataloaders` and the `paths` defined above. It uses the training augmentations for train and deterministic transforms for val/test, with seeded shuffling.

`test_loader` is created here for convenience but is **not used until Section 6**.

In [ ]:
set_seed(SEED)
train_loader, val_loader, test_loader = get_dataloaders(
    paths=paths,
    batch_size=HPARAMS["batch_size"],
    num_workers=HPARAMS["num_workers"],
    seed=SEED,
)

for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
    print(f"{name:5s}: {len(loader.dataset):5d} samples, {len(loader):3d} batches")

assert len(train_loader.dataset) == EXPECTED_SIZES["train"]
assert len(val_loader.dataset) == EXPECTED_SIZES["val"]
assert len(test_loader.dataset) == EXPECTED_SIZES["test"]

# Shape check on a validation batch (does not advance the training shuffle RNG).
images, labels = next(iter(val_loader))
print(f"\nBatch images: {tuple(images.shape)}  labels: {tuple(labels.shape)}  "
      f"label range: [{labels.min().item()}, {labels.max().item()}]")
assert images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)

## 3. Model

`create_resnet18_frozen()` loads ImageNet ResNet-18 (downloaded once, about 45 MB), replaces the final layer with `Linear(512, 26)`, then freezes every parameter except that new layer.

The checks below confirm, before any training, that:
1. only `fc.weight` and `fc.bias` are trainable, and every backbone parameter is frozen;
2. all 20 backbone BatchNorm layers stay in eval mode after `model.train()`, and their running statistics do not change during a training-mode forward pass;
3. the classifier outputs `[batch, 26]` with no NaN/Inf.

In [ ]:
set_seed(SEED)  # the new Linear(512, 26) layer is randomly initialised
model = create_resnet18_frozen(num_classes=NUM_CLASSES).to(device)

TOTAL_PARAMS = count_parameters(model)
TRAINABLE_PARAMS = count_parameters(model, trainable_only=True)
FROZEN_PARAMS = TOTAL_PARAMS - TRAINABLE_PARAMS

print(f"Model class:          {type(model).__name__} (ImageNet weights: {RESNET18_WEIGHTS.name})")
print(f"Classifier:           {model.fc}")
print(f"\nTotal parameters:     {TOTAL_PARAMS:,}")
print(f"Trainable parameters: {TRAINABLE_PARAMS:,} ({100 * TRAINABLE_PARAMS / TOTAL_PARAMS:.2f}%)")
print(f"Frozen parameters:    {FROZEN_PARAMS:,}")

# 1. Only the classifier is trainable; the whole backbone is frozen.
trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
frozen_names = [n for n, p in model.named_parameters() if not p.requires_grad]
assert isinstance(model.fc, nn.Linear) and (model.fc.in_features, model.fc.out_features) == (512, NUM_CLASSES)
assert trainable_names == ["fc.weight", "fc.bias"], trainable_names
assert all(not n.startswith("fc.") for n in frozen_names)
assert TRAINABLE_PARAMS == 512 * NUM_CLASSES + NUM_CLASSES
print(f"\nTrainable tensors:    {trainable_names}")
print(f"Frozen tensors:       {len(frozen_names)} (conv1, bn1, layer1-layer4)")

# 2. Frozen BatchNorm layers stay in eval mode during training and keep ImageNet statistics.
bn_layers = [m for m in model.modules() if isinstance(m, nn.BatchNorm2d)]
model.train()
assert model.training and model.fc.training
assert len(bn_layers) == 20 and not any(m.training for m in bn_layers), "frozen BatchNorm left eval mode"
bn_before = {k: v.clone() for k, v in model.state_dict().items() if "running_" in k or "num_batches" in k}
bn_images, _ = next(iter(val_loader))
with torch.no_grad():
    model(bn_images.to(device))  # training-mode forward pass, no weight update
bn_unchanged = all(torch.equal(bn_before[k], model.state_dict()[k]) for k in bn_before)
assert bn_unchanged, "BatchNorm running statistics changed in train mode"
model.eval()
print(f"BatchNorm layers:     {len(bn_layers)}, all in eval mode under model.train(): True")
print(f"BN running stats unchanged after a train-mode forward pass: {bn_unchanged}")


# 3. Forward-pass sanity check on a real validation batch (no gradient, no weight update).
sanity_images, _ = next(iter(val_loader))
with torch.no_grad():
    model.eval()
    sanity_out = model(sanity_images.to(device))
assert tuple(sanity_out.shape) == (sanity_images.shape[0], NUM_CLASSES), sanity_out.shape
assert torch.isfinite(sanity_out).all(), "model output contains NaN/Inf"
print(f"Input batch shape:  {list(sanity_images.shape)}")
print(f"Output shape:       {list(sanity_out.shape)}")
print(f"Output finite:      {torch.isfinite(sanity_out).all().item()} (no NaN/Inf)")

for d in (RUN_DIR, CHECKPOINT_DIR, FIGURE_DIR, METRICS_DIR):
    assert d.is_dir(), f"missing output directory {d}"
print(f"\nOutput directory:   {RUN_DIR} (exists)")
print(f"Best checkpoint ->  {CHECKPOINT_PATH}")

## 4. Training

`train_model` (from `src/training/train.py`):
- trains on `train_loader` and evaluates on `val_loader` after every epoch
- steps `ReduceLROnPlateau` on validation accuracy
- saves the best-validation checkpoint to `CHECKPOINT_PATH` whenever validation accuracy improves
- stops early after `early_stopping_patience` epochs without improvement
- restores the best-validation weights before returning

It has no test-loader argument, so the test split cannot influence training or model selection.

Only the classifier's parameters are given to the optimizer (`get_trainable_parameters`). `train_model` calls `model.train()` every epoch, and the frozen BatchNorm layers stay in eval mode automatically.

In [ ]:
criterion = nn.CrossEntropyLoss()
trainable_params = get_trainable_parameters(model)  # fc.weight, fc.bias only
assert sum(p.numel() for p in trainable_params) == TRAINABLE_PARAMS
optimizer = torch.optim.Adam(
    trainable_params, lr=HPARAMS["learning_rate"], weight_decay=HPARAMS["weight_decay"]
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=HPARAMS["scheduler_mode"],
    factor=HPARAMS["scheduler_factor"],
    patience=HPARAMS["scheduler_patience"],
)

history = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    num_epochs=HPARAMS["max_epochs"],
    device=device,
    scheduler=scheduler,
    criterion=criterion,
    monitor=HPARAMS["monitor"],
    early_stopping_patience=HPARAMS["early_stopping_patience"],
    checkpoint_path=CHECKPOINT_PATH,
    model_name=HPARAMS["model_name"],
    num_classes=NUM_CLASSES,
    seed=SEED,
    hyperparameters=HPARAMS,
)

# Persist the history immediately.
with open(METRICS_DIR / "history.json", "w") as f:
    json.dump(history, f, indent=2)
history_df = pd.DataFrame({k: history[k] for k in
                           ["epoch", "train_loss", "val_loss", "train_acc", "val_acc", "lr", "epoch_time"]})
history_df.to_csv(METRICS_DIR / "history.csv", index=False)

print(f"\nEpochs completed:   {history['epochs_completed']} (stopped early: {history['stopped_early']})")
print(f"Best epoch:         {history['best_epoch']}")
print(f"Best val accuracy:  {history['best_metric']:.4f}")
print(f"Total training time: {history['total_time']:.1f}s ({history['total_time'] / 60:.1f} min)")

### 4.1 Load the best validation checkpoint

The final model is reloaded from disk into a fresh frozen ResNet-18. This confirms the saved checkpoint is the selected model: its validation accuracy must match the best value recorded during training.

It also confirms the backbone really stayed frozen during training. Every backbone weight and BatchNorm statistic in the checkpoint must be identical to the original ImageNet weights; only `fc` may differ.

In [ ]:
best_model = create_resnet18_frozen(num_classes=NUM_CLASSES)
checkpoint = load_checkpoint(CHECKPOINT_PATH, model=best_model, map_location=device)
best_model.to(device).eval()

assert checkpoint["epoch"] == history["best_epoch"], "checkpoint is not from the best epoch"
assert checkpoint["num_classes"] == NUM_CLASSES

ckpt_val_loss, ckpt_val_acc = validate(best_model, val_loader, criterion, device)
print(f"Checkpoint epoch:   {checkpoint['epoch']}")
print(f"Validation acc:     {ckpt_val_acc:.4f} (recorded best: {history['best_metric']:.4f})")
assert abs(ckpt_val_acc - history["best_metric"]) < 1e-4, "reloaded checkpoint does not reproduce best val acc"

# Backbone unchanged by training: compare with the original ImageNet weights.
imagenet_reference = create_resnet18_frozen(num_classes=NUM_CLASSES).state_dict()
trained_state = best_model.state_dict()
changed = [
    k for k, v in imagenet_reference.items()
    if not k.startswith("fc.") and not torch.equal(v.cpu(), trained_state[k].cpu())
]
assert not changed, f"backbone tensors changed during training: {changed[:5]}"
print(f"Backbone tensors identical to ImageNet weights: "
      f"{sum(not k.startswith('fc.') for k in imagenet_reference)}/"
      f"{sum(not k.startswith('fc.') for k in imagenet_reference)} (only fc was trained)")
del imagenet_reference

## 5. Training Curves

In [ ]:
best_epoch = history["best_epoch"]


def plot_curve(train_key, val_key, ylabel, filename):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history["epoch"], history[train_key], marker="o", markersize=3, label="Train")
    ax.plot(history["epoch"], history[val_key], marker="o", markersize=3, label="Validation")
    ax.axvline(best_epoch, color="grey", linestyle="--", linewidth=1, label=f"Best epoch ({best_epoch})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(f"ResNet-18 frozen backbone — {ylabel}")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=150)
    plt.show()
    plt.close(fig)


plot_curve("train_loss", "val_loss", "Cross-entropy loss", "loss_curve.png")
plot_curve("train_acc", "val_acc", "Accuracy", "accuracy_curve.png")
print(f"Saved curves to {FIGURE_DIR}")

## 6. Test Evaluation

This is the first and only time the test split is used. The best-validation model is evaluated once, and nothing below feeds back into training or model selection.

`src/evaluation/` is not implemented yet, so predictions are collected with the small helper below and scored with scikit-learn.

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    y_true, y_pred, confidence = [], [], []
    for images, labels in loader:
        probs = torch.softmax(model(images.to(device, non_blocking=True)), dim=1)
        conf, pred = probs.max(dim=1)
        y_true.append(labels.numpy())
        y_pred.append(pred.cpu().numpy())
        confidence.append(conf.cpu().numpy())
    return np.concatenate(y_true), np.concatenate(y_pred), np.concatenate(confidence)


test_loss, _ = validate(best_model, test_loader, criterion, device)
y_true, y_pred, y_conf = predict(best_model, test_loader, device)
assert len(y_true) == EXPECTED_SIZES["test"]

labels_idx = list(range(NUM_CLASSES))
test_accuracy = accuracy_score(y_true, y_pred)
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels_idx, average="macro", zero_division=0
)
p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels_idx, average="weighted", zero_division=0
)

TEST_METRICS = {
    "test_loss": float(test_loss),
    "accuracy": float(test_accuracy),
    "precision_macro": float(p_macro),
    "recall_macro": float(r_macro),
    "f1_macro": float(f1_macro),
    "precision_weighted": float(p_weighted),
    "recall_weighted": float(r_weighted),
    "f1_weighted": float(f1_weighted),
    "num_test_samples": int(len(y_true)),
}
for k, v in TEST_METRICS.items():
    print(f"{k:20s} {v:.4f}" if isinstance(v, float) else f"{k:20s} {v}")

# Per-class report
report = classification_report(
    y_true, y_pred, labels=labels_idx, target_names=CLASS_NAMES, zero_division=0, output_dict=True
)
report_df = pd.DataFrame(report).T
report_df.to_csv(METRICS_DIR / "classification_report.csv")
print("\n" + classification_report(
    y_true, y_pred, labels=labels_idx, target_names=CLASS_NAMES, zero_division=0, digits=3
))

# Per-sample predictions (row order = test manifest order; the test loader is not shuffled)
test_df = test_loader.dataset.df
predictions_df = pd.DataFrame({
    "crop_path": test_df["crop_path"],
    "crop_filename": test_df["crop_filename"],
    "parent_filename": test_df["parent_filename"],
    "true_label": [CLASS_NAMES[i] for i in y_true],
    "pred_label": [CLASS_NAMES[i] for i in y_pred],
    "confidence": y_conf,
    "correct": y_true == y_pred,
})
predictions_df.to_csv(METRICS_DIR / "test_predictions.csv", index=False)
print(f"Misclassified test crops: {(~predictions_df['correct']).sum()} / {len(predictions_df)}")

## 7. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=labels_idx)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(METRICS_DIR / "confusion_matrix.csv")

row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums > 0)


def plot_confusion(matrix, title, filename, fmt):
    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(matrix, cmap="Blues")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(labels_idx, labels=CLASS_NAMES, rotation=75, ha="right", fontsize=8)
    ax.set_yticks(labels_idx, labels=CLASS_NAMES, fontsize=8)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(title)
    threshold = matrix.max() / 2 if matrix.max() > 0 else 0
    for i in labels_idx:
        for j in labels_idx:
            if matrix[i, j] > 0:
                ax.text(j, i, format(matrix[i, j], fmt), ha="center", va="center", fontsize=6,
                        color="white" if matrix[i, j] > threshold else "black")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=150)
    plt.show()
    plt.close(fig)


plot_confusion(cm, "ResNet-18 frozen backbone — test confusion matrix (counts)", "confusion_matrix.png", "d")
plot_confusion(cm_norm, "ResNet-18 frozen backbone — test confusion matrix (row-normalised recall)",
               "confusion_matrix_normalized.png", ".2f")
print(f"Saved confusion matrices to {FIGURE_DIR}")

## 8. Error Analysis

Uses only the test predictions from Section 6 (no retraining, no model selection):
- per-class precision, recall and F1, worst classes first
- the most frequent true → predicted confusions
- a gallery of misclassified test crops, most confident mistakes first

In [ ]:
ERROR_DIR = RUN_DIR / "error_analysis"
ERROR_DIR.mkdir(parents=True, exist_ok=True)

# Per-class metrics, worst F1 first
per_class_df = (
    report_df.loc[CLASS_NAMES, ["precision", "recall", "f1-score", "support"]]
    .rename(columns={"f1-score": "f1"})
    .astype({"support": int})
    .sort_values(["f1", "support"])
)
per_class_df.to_csv(ERROR_DIR / "per_class_metrics.csv", index_label="class_name")
print("Lowest-F1 classes:")
print(per_class_df.head(10).round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 9))
ax.barh(per_class_df.index, per_class_df["f1"], color="#4C72B0")
ax.set_xlim(0, 1)
ax.set_xlabel("F1-score (test)")
ax.set_title("ResNet-18 frozen backbone — per-class F1 on the test set")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "per_class_f1.png", dpi=150)
plt.show()
plt.close(fig)

# Most frequent confusions (off-diagonal confusion-matrix cells)
confusions = [
    {"true_label": CLASS_NAMES[i], "pred_label": CLASS_NAMES[j], "count": int(cm[i, j]),
     "share_of_true_class": float(cm[i, j] / cm[i].sum())}
    for i in labels_idx for j in labels_idx if i != j and cm[i, j] > 0
]
confused_pairs_df = pd.DataFrame(
    confusions, columns=["true_label", "pred_label", "count", "share_of_true_class"]
).sort_values("count", ascending=False)
confused_pairs_df.to_csv(ERROR_DIR / "confused_pairs.csv", index=False)
print("\nMost frequent confusions (true -> predicted):")
print(confused_pairs_df.head(10).round(3).to_string(index=False))

# Misclassified test crops, most confident mistakes first
misclassified_df = predictions_df[~predictions_df["correct"]].sort_values("confidence", ascending=False)
misclassified_df.to_csv(ERROR_DIR / "misclassified.csv", index=False)
print(f"\nMisclassified: {len(misclassified_df)} / {len(predictions_df)} test crops")

n_show = min(24, len(misclassified_df))
if n_show:
    n_cols = 6
    n_rows = int(np.ceil(n_show / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3.4 * n_rows))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, (_, row) in zip(np.atleast_1d(axes).ravel(), misclassified_df.head(n_show).iterrows()):
        with Image.open(resolve_crop_path(paths.processed_dir, row["crop_path"])) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(f"T: {row['true_label']}\nP: {row['pred_label']} ({row['confidence']:.2f})", fontsize=7)
    fig.suptitle("ResNet-18 frozen backbone — misclassified test crops (most confident first)")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "misclassified_examples.png", dpi=150)
    plt.show()
    plt.close(fig)

ERROR_SUMMARY = {
    "num_misclassified": int(len(misclassified_df)),
    "lowest_f1_classes": [
        {"class_name": c, "f1": float(r["f1"]), "support": int(r["support"])}
        for c, r in per_class_df.head(5).iterrows()
    ],
    "top_confusions": confused_pairs_df.head(5).to_dict(orient="records"),
}
print(f"\nSaved error analysis to {ERROR_DIR}")

## 9. Final Results

In [ ]:
best_idx = history["epoch"].index(history["best_epoch"])

RESULTS = {
    "experiment": "approach2_resnet18_frozen_backbone",
    "run_name": RUN_NAME,
    "git_commit": GIT_COMMIT,
    "seed": SEED,
    "hyperparameters": HPARAMS,
    "dataset": {
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "train_samples": len(train_loader.dataset),
        "val_samples": len(val_loader.dataset),
        "test_samples": len(test_loader.dataset),
        "zero_parent_overlap": True,
    },
    "model": {
        "name": HPARAMS["model_name"],
        "total_parameters": TOTAL_PARAMS,
        "trainable_parameters": TRAINABLE_PARAMS,
        "frozen_parameters": FROZEN_PARAMS,
        "pretrained_weights": HPARAMS["pretrained_weights"],
    },
    "training": {
        "epochs_completed": history["epochs_completed"],
        "stopped_early": history["stopped_early"],
        "best_epoch": history["best_epoch"],
        "best_val_acc": history["val_acc"][best_idx],
        "best_val_loss": history["val_loss"][best_idx],
        "train_acc_at_best_epoch": history["train_acc"][best_idx],
        "total_training_time_sec": history["total_time"],
        "mean_epoch_time_sec": float(np.mean(history["epoch_time"])),
    },
    "test": TEST_METRICS,
    "error_analysis": ERROR_SUMMARY,
    "hardware": HARDWARE,
    "artifacts": {
        "checkpoint": CHECKPOINT_PATH.relative_to(RUN_DIR).as_posix(),
        "history": "metrics/history.json",
        "classification_report": "metrics/classification_report.csv",
        "confusion_matrix": "metrics/confusion_matrix.csv",
        "test_predictions": "metrics/test_predictions.csv",
        "per_class_metrics": "error_analysis/per_class_metrics.csv",
        "confused_pairs": "error_analysis/confused_pairs.csv",
        "misclassified": "error_analysis/misclassified.csv",
        "figures": sorted(p.relative_to(RUN_DIR).as_posix() for p in FIGURE_DIR.glob("*.png")),
    },
}

with open(METRICS_DIR / "results.json", "w") as f:
    json.dump(RESULTS, f, indent=2)

summary = pd.Series({
    "Test accuracy": f"{TEST_METRICS['accuracy']:.4f}",
    "Precision (macro)": f"{TEST_METRICS['precision_macro']:.4f}",
    "Recall (macro)": f"{TEST_METRICS['recall_macro']:.4f}",
    "F1 (macro)": f"{TEST_METRICS['f1_macro']:.4f}",
    "F1 (weighted)": f"{TEST_METRICS['f1_weighted']:.4f}",
    "Best epoch": history["best_epoch"],
    "Best val accuracy": f"{RESULTS['training']['best_val_acc']:.4f}",
    "Training time": f"{history['total_time'] / 60:.1f} min",
    "Trainable parameters": f"{TRAINABLE_PARAMS:,} of {TOTAL_PARAMS:,}",
    "Hardware": HARDWARE["gpu_name"] or HARDWARE["device"],
})
print(summary.to_string())

print("\nREADME row:")
print(
    f"| ResNet-18 Frozen | {TEST_METRICS['accuracy']:.4f} | {TEST_METRICS['precision_macro']:.4f} | "
    f"{TEST_METRICS['recall_macro']:.4f} | {TEST_METRICS['f1_macro']:.4f} | "
    f"{history['total_time'] / 60:.1f} min |"
)
print(f"\nAll outputs saved under: {RUN_DIR}")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RUN_DIR)}  ({p.stat().st_size / 1024:.0f} KB)")